## Input File Requirements

This notebook evaluates spaCy Named Entity Recognition models using predictions generated in the first notebook and completed annotations exported from Label Studio.

Before running this notebook, you will need:

1. `tasks.json` – generated by `NER-exercise-public.ipynb`, containing the original tasks and model predictions.
2. `annotations.json` – exported from Label Studio after reviewing and completing the annotations.

These files are not included in this repository because they contain podcast transcript text. You will need to generate or obtain the corresponding files through the project workflow.

In [ ]:
import pandas as pd

In [ ]:
import json
from pathlib import Path

# Load the original predictions
with open("tasks.json", encoding="utf-8") as f:
    tasks = json.load(f)

# Load the completed Label Studio annotations
with open("annotations.json", encoding="utf-8") as f:
    annotations = json.load(f)

print("Original tasks:", len(tasks))
print("Completed annotations:", len(annotations))

## Inspect the structure of both files

In [ ]:
# Inspect the first original task
print("ORIGINAL TASK")
print(json.dumps(tasks[0], indent=2)[:3000])

# Inspect the first completed annotation
print("\nCOMPLETED ANNOTATION")
print(json.dumps(annotations[0]["annotations"][0], indent=2)[:3000])

## Verify the models' predictions

In [ ]:
# Extract predictions from the first task
for prediction in tasks[0]["predictions"]:

    print("\nModel:", prediction["model_version"])

    for entity in prediction["result"]:
        value = entity["value"]

        print(
            value["text"],
            "—",
            value["labels"][0],
            f"({value['start']}, {value['end']})"
        )

##  Function to extract entities

In [ ]:
def extract_entities(results):
    entities = []

    for item in results:
        value = item["value"]

        entities.append({
            "text": value["text"],
            "label": value["labels"][0],
            "start": value["start"],
            "end": value["end"]
        })

    return entities

## Testing function on the small model's first prediction

In [ ]:
small_entities = extract_entities(
    tasks[0]["predictions"][0]["result"]
)

small_entities

## Extract the large model's predictions using the function

In [ ]:
large_entities = extract_entities(
    tasks[0]["predictions"][1]["result"]
)

large_entities

## Extract your human annotations using the function

In [ ]:
human_entities = extract_entities(
    annotations[0]["annotations"][0]["result"]
)

human_entities

## Compare the three datasets for this transcript

In [ ]:
first_task_comparison = {
    "small_model": small_entities,
    "large_model": large_entities,
    "human_reference": human_entities
}

for name, entities in first_task_comparison.items():
    print(f"\n{name}: {len(entities)} entities")
    for entity in entities:
        print(f"  {entity['text']} — {entity['label']}")

## Extracting entities from each transcript in a list called evaluation_data

In [ ]:
evaluation_data = []

for original, completed in zip(tasks, annotations):

    # Get the transcript
    text = original["data"]["text"]

    # Extract predictions from both models
    small_prediction = extract_entities(
        original["predictions"][0]["result"]
    )

    large_prediction = extract_entities(
        original["predictions"][1]["result"]
    )

    # Extract the completed human annotation
    human_reference = extract_entities(
        completed["annotations"][0]["result"]
    )

    # Store everything together
    evaluation_data.append({
        "text": text,
        "small_model": small_prediction,
        "large_model": large_prediction,
        "human_reference": human_reference
    })

print("Total transcripts:", len(evaluation_data))

## Inspect the result

In [ ]:
evaluation_data[0]

## Check the human annotation

In [ ]:
boundary_issues = []

for i, task in enumerate(evaluation_data):

    text = task["text"]

    for entity in task["human_reference"]:

        extracted_text = text[entity["start"]:entity["end"]]

        if extracted_text != entity["text"]:
            boundary_issues.append({
                "task": i,
                "entity": entity["text"],
                "extracted": extracted_text,
                "issue": "Text does not match character positions"
            })

        if entity["text"] != entity["text"].strip():
            boundary_issues.append({
                "task": i,
                "entity": entity["text"],
                "extracted": extracted_text,
                "issue": "Leading or trailing whitespace"
            })

print("Total boundary issues:", len(boundary_issues))

for issue in boundary_issues:
    print(issue)

## Correcting the boundaries in our working dataset

In [ ]:
# Create a cleaned copy of the human annotations
for task in evaluation_data:

    cleaned_entities = []

    for entity in task["human_reference"]:

        cleaned_entity = entity.copy()

        # Count leading and trailing whitespace
        leading_spaces = len(cleaned_entity["text"]) - len(
            cleaned_entity["text"].lstrip()
        )

        trailing_spaces = len(cleaned_entity["text"]) - len(
            cleaned_entity["text"].rstrip()
        )

        # Adjust the character positions
        cleaned_entity["start"] += leading_spaces
        cleaned_entity["end"] -= trailing_spaces

        # Remove whitespace around the entity text
        cleaned_entity["text"] = cleaned_entity["text"].strip()

        cleaned_entities.append(cleaned_entity)

    task["human_reference_clean"] = cleaned_entities

print("Cleaned human annotations created.")

# Verify the corrections

In [ ]:
cleaned_boundary_issues = []

for i, task in enumerate(evaluation_data):

    text = task["text"]

    for entity in task["human_reference_clean"]:

        extracted_text = text[entity["start"]:entity["end"]]

        if extracted_text != entity["text"]:
            cleaned_boundary_issues.append({
                "task": i,
                "entity": entity["text"],
                "extracted": extracted_text
            })

print("Remaining boundary issues:", len(cleaned_boundary_issues))

## Prepare Evaluation

In [ ]:
# Count entities across all 34 transcripts

total_small = sum(
    len(task["small_model"]) for task in evaluation_data
)

total_large = sum(
    len(task["large_model"]) for task in evaluation_data
)

total_human = sum(
    len(task["human_reference_clean"]) for task in evaluation_data
)

print("Small model entities:", total_small)
print("Large model entities:", total_large)
print("Human reference entities:", total_human)

## Calculate entities for small model results

In [ ]:
# Convert entities into sets of comparable tuples

small_set = {
    (e["start"], e["end"], e["label"])
    for e in evaluation_data[0]["small_model"]
}

large_set = {
    (e["start"], e["end"], e["label"])
    for e in evaluation_data[0]["large_model"]
}

human_set = {
    (e["start"], e["end"], e["label"])
    for e in evaluation_data[0]["human_reference_clean"]
}

# Calculate TP, FP and FN for the small model

small_tp = len(small_set & human_set)
small_fp = len(small_set - human_set)
small_fn = len(human_set - small_set)

print("Small model")
print("TP:", small_tp)
print("FP:", small_fp)
print("FN:", small_fn)

## Calculate large model's results

In [ ]:
large_tp = len(large_set & human_set)
large_fp = len(large_set - human_set)
large_fn = len(human_set - large_set)

print("Large model")
print("TP:", large_tp)
print("FP:", large_fp)
print("FN:", large_fn)

## Automate the comparison across all 34 transcripts


In [ ]:
def calculate_counts(predictions, reference):

    predicted_set = {
        (e["start"], e["end"], e["label"])
        for e in predictions
    }

    reference_set = {
        (e["start"], e["end"], e["label"])
        for e in reference
    }

    tp = len(predicted_set & reference_set)
    fp = len(predicted_set - reference_set)
    fn = len(reference_set - predicted_set)

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn
    }

## Test the calculate_counts() function

In [ ]:
print("Small model:")
print(calculate_counts(
    evaluation_data[0]["small_model"],
    evaluation_data[0]["human_reference_clean"]
))

print("\nLarge model:")
print(calculate_counts(
    evaluation_data[0]["large_model"],
    evaluation_data[0]["human_reference_clean"]
))

## Calculate results for all 34 transcripts using function

In [ ]:
all_results = []

for i, task in enumerate(evaluation_data):

    human = task["human_reference_clean"]

    small_counts = calculate_counts(
        task["small_model"],
        human
    )

    large_counts = calculate_counts(
        task["large_model"],
        human
    )

    all_results.append({
        "task": i + 1,
        "small_model": small_counts,
        "large_model": large_counts
    })

print("Transcripts evaluated:", len(all_results))

## Inspect the first five results

In [ ]:
for result in all_results[:5]:

    print(f"\nTranscript {result['task']}")

    print("Small model:", result["small_model"])
    print("Large model:", result["large_model"])

## Aggregate the results across all transcripts

In [ ]:
# Aggregate results for both models

small_total = {
    "TP": sum(r["small_model"]["TP"] for r in all_results),
    "FP": sum(r["small_model"]["FP"] for r in all_results),
    "FN": sum(r["small_model"]["FN"] for r in all_results)
}

large_total = {
    "TP": sum(r["large_model"]["TP"] for r in all_results),
    "FP": sum(r["large_model"]["FP"] for r in all_results),
    "FN": sum(r["large_model"]["FN"] for r in all_results)
}

print("SMALL MODEL:", small_total)
print("LARGE MODEL:", large_total)

## Calculating precision, recall and F1

In [ ]:
def calculate_metrics(counts):

    tp = counts["TP"]
    fp = counts["FP"]
    fn = counts["FN"]

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    return {
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }


small_metrics = calculate_metrics(small_total)
large_metrics = calculate_metrics(large_total)

print("SMALL MODEL")
for metric, score in small_metrics.items():
    print(f"{metric}: {score:.4f} ({score * 100:.2f}%)")

print("\nLARGE MODEL")
for metric, score in large_metrics.items():
    print(f"{metric}: {score:.4f} ({score * 100:.2f}%)")

In [ ]:
%pip install matplotlib

## Graph to compare scores of Small and Large Model

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Metric names
metrics = list(small_metrics.keys())

# Convert scores to percentages
small_scores = [score * 100 for score in small_metrics.values()]
large_scores = [score * 100 for score in large_metrics.values()]

# Set bar positions and width
x = np.arange(len(metrics))
width = 0.35

# Create the chart
fig, ax = plt.subplots(figsize=(9, 5))

small_bars = ax.bar(
    x - width/2, small_scores, width,
    label="Small Model"
)

large_bars = ax.bar(
    x + width/2, large_scores, width,
    label="Large Model"
)

# Add labels and title
ax.set_title("spaCy Small vs Large Model Performance", fontsize=14)
ax.set_ylabel("Score (%)")
ax.set_xlabel("Evaluation Metrics")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 100)

# Display scores above each bar
ax.bar_label(small_bars, fmt="%.2f%%", padding=3)
ax.bar_label(large_bars, fmt="%.2f%%", padding=3)

# Add legend and grid
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()
